# California Housing Price Regression Analysis and Deployment

In [51]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.graphics.gofplots import qqplot
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import numpy as np
import joblib


## Step 1: Load the Data

In [52]:
df = pd.read_csv("housing.csv")  # Adjust path if needed

## Step 2: Exploratory Data Analysis (EDA)

In [53]:

print(df.info())
print(df.describe())

# Histograms
plt.figure(figsize=(16, 12))
df.hist(bins=30, edgecolor='black', figsize=(16, 12))
plt.suptitle("Feature Distributions", fontsize=18)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# Correlation heatmap
corr_matrix = df.drop(columns=['ocean_proximity']).corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Matrix")
plt.show()

# Scatter plots
important_features = ['median_income', 'housing_median_age', 'total_rooms', 'latitude', 'longitude']
plt.figure(figsize=(16, 12))
for i, feature in enumerate(important_features):
    plt.subplot(3, 2, i + 1)
    sns.scatterplot(x=df[feature], y=df['median_house_value'], alpha=0.4)
    plt.title(f"{feature} vs. median_house_value")
    plt.xlabel(feature)
    plt.ylabel("Median House Value")
plt.tight_layout()
plt.show()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB
None
          longitude      latitude  housing_median_age   total_rooms  \
count  20640.000000  20640.000000        20640.000000  20640.000000   
mean    -119.569704     35.631861           28.639486   2635.763081   
std        2.003532      2.135952

C:\Users\ahmed\AppData\Local\Temp\ipykernel_30456\933751357.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\ahmed\AppData\Local\Temp\ipykernel_30456\933751357.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\ahmed\AppData\Local\Temp\ipykernel_30456\933751357.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 3: Data Cleaning

In [54]:

print("Missing values:\n", df.isnull().sum())
df['total_bedrooms'].fillna(df['total_bedrooms'].median(), inplace=True)
df['ocean_proximity'] = df['ocean_proximity'].astype('category')


Missing values:
 longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64


C:\Users\ahmed\AppData\Local\Temp\ipykernel_30456\3373746534.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['total_bedrooms'].fillna(df['total_bedrooms'].median(), inplace=True)


## Step 4: Model Development

In [55]:

df_encoded = pd.get_dummies(df, drop_first=True)
X = df_encoded.drop('median_house_value', axis=1)
y = df_encoded['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

# Evaluation
print("\nModel Evaluation:")
print(f"MSE: {mean_squared_error(y_test, y_pred):.2f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"R²: {r2_score(y_test, y_pred):.2f}")

# Save model and scaler
joblib.dump(model, "linear_model.pkl")
joblib.dump(scaler, "scaler.pkl")



Model Evaluation:
MSE: 4908476721.16
MAE: 50670.74
R²: 0.63


['scaler.pkl']

## Step 5: Regression Diagnostics

In [56]:

residuals = y_test - y_pred

# Residual plot
plt.figure(figsize=(8, 5))
sns.scatterplot(x=y_pred, y=residuals)
plt.axhline(0, color='red', linestyle='--')
plt.title("Residuals vs. Predicted")
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.show()

# Q-Q plot
qqplot(residuals, line='s')
plt.title("Q-Q Plot of Residuals")
plt.show()

# VIF
X_vif = sm.add_constant(X_train_scaled)
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif, i+1) for i in range(X.shape[1])]
print("\nVariance Inflation Factors:")
print(vif_data)



Variance Inflation Factors:
                       Feature        VIF
0                    longitude  18.000488
1                     latitude  19.868698
2           housing_median_age   1.321783
3                  total_rooms  13.078025
4               total_bedrooms  36.638126
5                   population   6.429272
6                   households  35.864528
7                median_income   1.797340
8       ocean_proximity_INLAND   2.849961
9       ocean_proximity_ISLAND   1.002189
10    ocean_proximity_NEAR BAY   1.565865
11  ocean_proximity_NEAR OCEAN   1.193029


C:\Users\ahmed\AppData\Local\Temp\ipykernel_30456\415588804.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\ahmed\AppData\Local\Temp\ipykernel_30456\415588804.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [57]:
import joblib

joblib.dump(train_encoded_cols, "train_encoded_cols.pkl")


['train_encoded_cols.pkl']

## Step 6: Streamlit Web App (app.py)

In [59]:
# After you create X for training
X = df_encoded.drop('median_house_value', axis=1)
joblib.dump(list(X.columns), "train_encoded_cols.pkl")

['train_encoded_cols.pkl']

In [60]:
import streamlit as st
import pandas as pd
import joblib

model = joblib.load("linear_model.pkl")
scaler = joblib.load("scaler.pkl")
train_encoded_cols = joblib.load("train_encoded_cols.pkl")  # This is the list you saved during training

# ...collect user input...

input_data = pd.DataFrame(input_dict)
input_encoded = pd.get_dummies(input_data, columns=['ocean_proximity'])

# Add missing columns and ensure order
for col in train_encoded_cols:
    if col not in input_encoded.columns:
        input_encoded[col] = 0

# Remove any extra columns and ensure correct order
input_encoded = input_encoded[train_encoded_cols]

# Now you can safely scale and predict
scaled_input = scaler.transform(input_encoded)
prediction = model.predict(scaled_input)